# ModernBERT - Run the Pipeline from GitHub

One-shot entry point for the `Compartment` repo (`main.py` + `src/` package).

**How to use**

1. Edit the cell below: set `GITHUB_URL` to your repository (only needed if you are not using a GitHub input) and decide `USE_KAGGLE_INPUT`.
2. Make sure the **Compartment** dataset is attached as an input (it provides `dataset/` and `trial/`). 
3. Click **Run All**.

**What happens**

- locates/clones the repo (GitHub input mount or `git clone`)
- ensures the Python environment is ready
- runs `main.py <MODE>` through the CLI (config + validation happen there)
- prints metrics, previews the submission and creates `submission.zip`


## 1. Locate / clone the repository
Set the two variables at the top of this cell, then run it.


In [ ]:
import glob, json, os, subprocess, sys, zipfile

# ==== 1. Point at your repo ==========================================
USE_KAGGLE_INPUT = True   # True  -> GitHub repo mounted via "Add Input -> GitHub"
                          # False -> git clone the URL below into /kaggle/working
GITHUB_URL = 'https://github.com/AmnO-O/MoTune.git'   # <-- EDIT ME

# ==== 2. Locate the repo root ========================================
def sh(cmd, cwd=None):
    print('>>>', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    sys.stdout.write(r.stdout)
    sys.stderr.write(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')

REPO = None
if USE_KAGGLE_INPUT:
    mains = sorted(glob.glob('/kaggle/input/github/**/main.py', recursive=True))
    if mains:
        REPO = os.path.dirname(mains[0])   # read-only snapshot; Kaggle input is fixed
if REPO is None:
    REPO = '/kaggle/working/compartment'
    if os.path.isdir(os.path.join(REPO, '.git')):
        # in-session clone from an earlier run -> refresh to the latest commit
        sh(f'git -C {REPO} fetch --depth 1 origin main')
        sh(f'git -C {REPO} reset --hard origin/main')
    elif not os.path.isdir(os.path.join(REPO, 'src')):
        sh(f'git clone --depth 1 {GITHUB_URL} {REPO}')

print('REPO =', REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)
print('CWD =', os.getcwd())


## 2. Environment
Only installs on Kaggle if a dependency is missing (they all ship pre-installed).


In [ ]:
import subprocess, sys

# On Kaggle, torch / transformers are preinstalled. Only install when missing.
check = subprocess.run(
    [sys.executable, '-c', 'import torch, transformers, pandas, numpy, sklearn, scipy'],
    capture_output=True,
)
if check.returncode != 0:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch', 'transformers>=4.41', 'pandas', 'numpy', 'scikit-learn', 'scipy'],
        check=True,
    )
    print('Installed missing packages.')
else:
    print('Environment OK (torch, transformers, pandas, numpy, sklearn, scipy).')


## 3. Choose the run
Set `MODE` to `train80`, `train5` or `predict`. Append any CLI override to
`EXTRA_TRAIN` / `EXTRA_PREDICT` (e.g. `' --epochs 8 --batch 16 --ccc-weight 0.5'`).


In [ ]:
# ==== 3. The run ======================================================
MODE = 'train80'         # 'train80' = quick 80/20 split
                         # 'train5'  = stratified 5-fold CV (default)
                         # 'predict' = trial predictions + submission
DO_PREDICT = True        # run "main.py predict" after training (needs checkpoints)
EXTRA_TRAIN = ''         # optional CLI overrides, e.g. ' --epochs 8 --batch 16'
EXTRA_PREDICT = ''       # e.g. ' --predict-mode single --seed 7'
                         # predict-mode is auto-derived: 'single' after train80,
                         # '5fold' after train5 (only override if you know better)

# Data location is auto-detected (Kaggle dataset input or ./dataset) but can be
# pinned with ' --data-path <dir>'. Outputs go to /kaggle/working (or ./output).


## 3b. Config overrides (optional)

Edit the `OVERRIDES` dict below to change any hyperparameter. The cell
validates it strictly against `config.py` (a typo'd key fails fast, before the
GPU hours start) and writes `config_run.json`, which is passed to
`main.py --config`. CLI flags in `EXTRA_TRAIN` / `EXTRA_PREDICT` still win over
this file. Leave the dict empty to run on pure defaults.

**Important:** `train` and `predict` must share the same model knobs
(`model_name`, `head_mode`, `num_bins`) - the same config file is used for both.

In [ ]:
# ==== 3b. Config overrides ==============================================
# Uncomment the lines you want to change. Keys must match Config fields in
# config.py. EVERYTHING is optional.
OVERRIDES = {
    # 'model_name': 'answerdotai/ModernBERT-base',
    # 'hidden_size': 768,              # backbone hidden dim \n    # 'dropout': 0.2,
    # 'head_mode': 'reg',              # 'reg' or 'softmax' (ordinal bins -> E[Y])
    # 'num_bins': 6,                   # ordinal bins, centers uniform in [1, 5]
    # 'max_length': 128,
    # 'max_context_length': 256,
    # 'batch_size': 32,                # micro-batch
    # 'accum_steps': 1,                # effective batch = batch_size * accum_steps
    # 'num_epochs': 10,
    # 'freeze_epochs': 5,              # Phase 1 length (encoder frozen)
    # 'unfreeze_from_layer': 19,       # first encoder layer unfrozen in Phase 2
    # 'head_lr': 5e-4,
    # 'encoder_lr': 3e-6,
    # 'embedding_lr': 1e-5,
    # 'warmup_ratio': 0.15,
    # 'ccc_weight': 0.7,               # blend of MSE and (1 - CCC)
    # 'lambda_rank': 0.0,              # >0 adds pairwise margin-ranking loss
    # 'rank_margin': 0.5,
    # 'ce_weight': 0.0,                # >0 adds Gaussian soft-target CE in softmax mode
    # 'bin_sigma': 0.5,                # soft-target Gaussian width (bin units)
    # 'patience': 5,
    # 'n_splits': 5,                   # folds for train5
    # 'test_size': 0.2,                # val fraction for train80
    # 'predict_mode': 'single',        # 'single' or '5fold' \n    # 'seed': 42,
}

import json, os, sys

CONFIG_PATH = os.path.join(os.getcwd(), 'config_run.json')
if not OVERRIDES:
    if os.path.isfile(CONFIG_PATH):
        os.remove(CONFIG_PATH)
    print('No overrides - running on defaults (config_run.json removed).')
else:
    from config import Config
    cfg = Config.defaults().update(**OVERRIDES)
    cfg.validate()
    json.dump(OVERRIDES, open(CONFIG_PATH, 'w'), indent=2)
    print('Config OK and written to', CONFIG_PATH)
    print('Active overrides:', json.dumps(OVERRIDES, indent=2))

In [ ]:
import os, shlex, subprocess, sys

cfg_arg = ' --config ' + CONFIG_PATH if os.path.isfile(CONFIG_PATH) else ''

PY = sys.executable

if MODE not in ('train80', 'train5', 'predict'):
    raise SystemExit(f'Unknown MODE {MODE!r} (use train80 / train5 / predict)')

def run(cmd: str):
    print('>>>', cmd)
    subprocess.run(shlex.split(cmd), check=True, cwd=os.getcwd())

if MODE == 'predict':
    base = EXTRA_PREDICT or '--predict-mode single'
    run(f'{PY} main.py predict {base}{cfg_arg}')
else:
    run(f'{PY} main.py {MODE} {EXTRA_TRAIN}{cfg_arg}')
    if DO_PREDICT:
        # '5fold' needs fold{0..4}_best.pt (written by train5); train80 only
        # writes best.pt, so predict must use 'single' after train80.
        pm = EXTRA_PREDICT
        if 'predict-mode' not in pm:
            auto = 'single' if MODE == 'train80' else '5fold'
            pm = f'{pm.strip()} --predict-mode {auto}'.strip()
        print('\n--- now generating the submission ---')
        run(f'{PY} main.py predict {pm}{cfg_arg}')


## 4. Results
Printed from the JSON artifacts that `main.py` writes next to the checkpoints.


In [ ]:
import json, os

OUT = '/kaggle/working'
for fname in ('metrics.json', 'trial_metrics.json'):
    path = os.path.join(OUT, fname)
    if os.path.isfile(path):
        print('=' * 20, fname, '=' * 20)
        print(json.dumps(json.load(open(path)), indent=2))


## 5. Submission
Preview the predictions and get a Kaggle-ready `submission.zip` (TSV, no header).


In [ ]:
import os, zipfile

import pandas as pd

sub = os.path.join('/kaggle/working', 'submission', 'en-nn-trial-pred.tsv')
if not os.path.isfile(sub):
    print('No submission yet - run with MODE=predict or DO_PREDICT=True.')
else:
    df = pd.read_csv(sub, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
    print(df.head(10).to_string(index=False))
    print(f'\nTotal rows: {len(df)}')

    zip_path = os.path.join('/kaggle/working', 'submission', 'submission.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(sub, arcname=os.path.basename(sub))
    print('Kaggle-ready submission:', zip_path)
